# Phase 5: Legal Statute Identification - Legal Transformer with LoRMA

## Task
Legal statute identification: read a case and predict which statutes apply. A case can have several statutes (multi-label). Scored with macro F1.

Model: the pre-trained Legal Transformer fine-tuned with LoRMA. Each case is cut into chunks, each chunk becomes a vector, and a head gives one score per statute.

## Input
- Phase-5 data (train, dev, test cases); downloaded from Hugging Face if not attached.
- Legal Transformer checkpoint and tokenizer.

## Flow of the notebook (from input to output)
1. Set the file paths and the training settings.
2. Write the main pipeline lsi_core.py: data loading, cutting long cases into chunks, the document model (chunk encoder and head), training, threshold tuning and submission writing.
3. Write the fine-tuning helpers lsi_peft.py (freezing, LoRMA, layer-wise learning rates, memory-saving blocks).
4. Write the Legal Transformer model code.
5. Load the model, set up LoRMA, and run the pipeline: download the data, tokenize and chunk the cases, train with evaluation on dev, tune one threshold per statute on dev, and predict the test cases (every case gets at least one statute).

## Output
- Submission zips with the predicted statutes per test case.
- Summary of dev scores and one zip with all outputs.

In [1]:
# Settings: file paths and training config
CKPT = "/kaggle/input/datasets/patelpreeet/final-pt-lorma-legal-tranformer/legal_transformer_random_moredata_best_random_90_10.pt"
TOK = "/kaggle/input/datasets/patelpreeet/tokenizer-transformer-lorma/tokenizer_legal_transformer_random_moredata_random_90_10.json"
import os, glob
# It takes path (file path) and pats (file patterns) as input, uses the given path, or searches the input folder for the file and returns a file path.
def _find(path, pats):
    if os.path.exists(path):
        return path
    for p in pats:
        h = sorted(x for x in glob.glob("/kaggle/input/**/" + p, recursive=True) if "resume" not in x and "trainable" not in x and os.path.isfile(x))
        if h:
            return h[0]
    return path
CKPT = _find(CKPT, ['*legal*transformer*.pt', '*lt*.pt', '*legal*transformer*/**/*.pt', '*lt*/**/*.pt'])
TOK = _find(TOK, ['*legal*transformer*.json', '*lt*.json', '*legal*transformer*/**/*.json', '*lt*/**/*.json', '*tok*legal*transformer*.json', '*tok*lt*.json'])
print(CKPT)
print(TOK)
for _p in (CKPT, TOK):
    if not os.path.isfile(_p):
        print("missing:", _p)
        print([x for x in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(x)][:80])
        raise FileNotFoundError(_p)
OUT = "/kaggle/working"
NAME = "p5_1_lt_lorma_opt"
HEAD_LR = 1e-3
PEFT_LR = 4e-4
R = 16
ALPHA = 32
TOP_K = 12
CFG = {'W': 576, 'maxch': 16, 'first': 8, 'k_lo': 4, 'k_hi': 8, 'batch_docs': 16, 'mb': 4, 'epochs': 3, 'pw_power': 0.5, 'pw_max': 30.0, 'loss': 'bce', 'tok_mask': 0.05, 'head_dropout': 0.3, 'hid': 384, 'gru': 256, 'warmup': 0.05, 'clip': 1.0, 'wd': 0.01, 'ema': 0.999, 'evals': 12, 'eval_ch': 6, 'eval_docs': 1500, 'max_hours': 11.0, 'session_hours': 11.0, 'train_frac': 0.62, 'stop_frac': 0.72, 'save_every': 900, 'sanity_step': 150, 'init_scale': 4096.0, 'cal_steps': 30, 'cb': 48, 'seed': 0, 'kd_alpha': 0.0, 'kd_T': 2.0, 'dump_train': True, 'smoke': 0, 'lp_docs': 3000, 'lp_epochs': 20, 'refit_seeds': 2, 'refit_losses': ('bce', 'asl'), 'refit_epochs': 25, 'chunk_drop': 0.2, 'ramp_steps': 150, 'patience': 5, 'max_rollbacks': 3, 'drop_tol': 0.03, 'skip_train': False}
SMOKE = 0

/kaggle/input/datasets/patelpreeet/final-pt-lorma-legal-tranformer/legal_transformer_random_moredata_best_random_90_10.pt
/kaggle/input/datasets/patelpreeet/tokenizer-transformer-lorma/tokenizer_legal_transformer_random_moredata_random_90_10.json


In [2]:
%%writefile lsi_core.py
# Main statute-identification code: data, model, training (saved as lsi_core.py)
import os, re, json, glob, math, time, random, copy, zipfile, pickle, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO = "Exploration-Lab/CS-781-Capstone"
T0, NPOS = time.time(), 6
GRID = np.arange(0.02, 0.96, 0.02)
TEXT_KEYS = ("text", "case_text", "document", "content", "case")
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "y")

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    hits = [h for r in roots for h in glob.glob(os.path.join(r, "**", name), recursive=True)
            if "__macosx" not in h.lower() and any(k in h.lower() for k in ("phase_5", "phase5", "lsi"))]
    return min(hits, key=len) if hits else None

# It takes dest (download folder) as input, downloads the data if it is not attached and returns a dictionary from file name to path.
def ensure_data(dest="/kaggle/working/phase5_data"):
    names = ["train.json", "dev.json", "test.json"]
    if all(find_file(n) for n in names):
        return {n: find_file(n) for n in names}
    from huggingface_hub import HfApi, hf_hub_download, login
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    login(token=tok)
    files = HfApi().list_repo_files(REPO, repo_type="dataset")
    out = {}
    for n in names:
        c = [f for f in files if f.lower().endswith("/" + n) and ("phase_5" in f.lower() or "lsi" in f.lower())]
        assert c, f"{n} not found; files: {files[:20]}"
        out[n] = hf_hub_download(REPO, min(c, key=len), repo_type="dataset", local_dir=dest)
    return out

# It takes path (file path) as input, loads documents from a JSON file and returns a list of documents.
def load_docs(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    print("fields of the first record:", list(raw[0].keys()), flush=True)
    out = []
    for it in raw:
        t = next(it[k] for k in TEXT_KEYS if k in it)
        t = " ".join(map(str, t)) if isinstance(t, list) else str(t)
        y = next((it[k] for k in LABEL_KEYS if k in it and it[k] is not None), None)
        if isinstance(y, str):
            y = re.findall(r"\d+", y)
        y = None if y is None else sorted({int(float(v)) for v in (y if isinstance(y, (list, tuple)) else [y])})
        out.append(dict(id=it.get("id", it.get("case_id")), text=re.sub(r"\s+", " ", t).strip(), y=y))
    return out

_TK, _DT = None, np.int32

# It takes text as input, tokenizes one text and returns an array of token ids.
def _tok_one(text):
    return np.asarray(_TK.encode(text), dtype=_DT)

class Doc:
    __slots__ = ("id", "y", "head", "tail", "n", "segs", "pos", "soft")

    # It takes i (index) as input, returns the tokens of one chunk.
    def chunk(self, i):
        a, s, e = self.segs[i]
        return (self.head if a == 0 else self.tail)[s:e]

# It takes id_ (document id), ids (token ids), W (chunk size), maxch (max chunks), first (number of first chunks) and eos (end token id) as input, cuts a long document into chunks and returns a Doc object with chunks.
def build_doc(id_, ids, W, maxch, first, eos):
    ids = ids if len(ids) else np.asarray([eos], _DT)
    n = len(ids)
    tail, head = (ids, ids[:0]) if n <= maxch * W else (ids[-(maxch - first) * W:], ids[:first * W])
    d = Doc(); d.id, d.y, d.head, d.tail, d.n, d.soft = id_, None, head, tail, n, None
    segs, bounds = [], []
    for s in range(0, len(head), W):
        e = min(s + W, len(head)); segs.append((0, s, e)); bounds.append((s, e))
    t, e = [], len(tail)
    while e > 0:
        s = max(0, e - W); t.append((1, s, e)); e = s
    off = n - len(tail)
    for a, s, e in t[::-1]:
        segs.append((a, s, e)); bounds.append((off + s, off + e))
    m = len(segs)
    pos = np.zeros((m, NPOS), np.float32)
    for i, (s, e) in enumerate(bounds):
        k = m - 1 - i
        pos[i] = (s / n, (n - e) / n, math.log1p(k) / 3, math.log1p(n / W) / 3, float(k == 0), float(i == 0))
    d.segs, d.pos = segs, pos
    return d

# It takes raw (raw documents), tok (tokenizer), cfg (settings) and cache (cache file path) as input, tokenizes all documents, using a cache and returns a list of Doc objects.
def tokenize_docs(raw, tok, cfg, cache):
    global _TK, _DT
    _DT = np.uint16 if getattr(tok, "vocab_size", 10**6) < 65535 else np.int32
    key = hashlib.md5(json.dumps([len(raw), cfg["W"], cfg["maxch"], cfg["first"], raw[0]["id"], raw[-1]["id"], len(raw[0]["text"]), len(raw[-1]["text"])], default=str).encode()).hexdigest()[:10]
    path = f"{cache}.{key}.pkl"
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    _TK = tok
    t0 = time.time()
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(min(4, os.cpu_count() or 1)) as pool:
            ids = pool.map(_tok_one, [r["text"] for r in raw], chunksize=32)
    except Exception:
        ids = [_tok_one(r["text"]) for r in raw]
    docs = [build_doc(r["id"], a, cfg["W"], cfg["maxch"], cfg["first"], tok.eos_id) for r, a in zip(raw, ids)]
    print(f"tokenised {len(docs)} documents in {time.time() - t0:.0f}s, mean {np.mean([d.n for d in docs]):.0f} tokens", flush=True)
    pickle.dump(docs, open(path, "wb"))
    return docs

# It takes items (list of items), pad (padding id), dev (device), tok_mask (token masking rate) and rng (random generator) as input, pads chunks into one batch and returns a dictionary of tensors.
def collate(items, pad, dev, tok_mask=0.0, rng=None):
    arrs, bi, ki = [], [], []
    for b, (d, ix) in enumerate(items):
        for j, i in enumerate(ix):
            arrs.append(d.chunk(i)); bi.append(b); ki.append(j)
    N, T = len(arrs), max(len(a) for a in arrs)
    ids = np.full((N, T), pad, np.int64); lens = np.empty(N, np.int64)
    for r, a in enumerate(arrs):
        ids[r, :len(a)] = a; lens[r] = len(a)
    if tok_mask > 0 and rng is not None:
        valid = np.arange(T)[None] < lens[:, None]
        src = np.minimum((rng.random(ids.shape) * lens[:, None]).astype(np.int64), T - 1)
        ids = np.where((rng.random(ids.shape) < tok_mask) & valid, np.take_along_axis(ids, src, 1), ids)
    B, km = len(items), max(len(ix) for _, ix in items)
    pos = np.zeros((B, km, NPOS), np.float32); cm = np.zeros((B, km), bool)
    for b, (d, ix) in enumerate(items):
        pos[b, :len(ix)] = d.pos[ix]; cm[b, :len(ix)] = True
    t = lambda a: torch.from_numpy(a).to(dev)
    return dict(ids=t(ids), lens=t(lens), bi=t(np.array(bi)), ki=t(np.array(ki)), pos=t(pos), cm=t(cm))

class Enc(nn.Module):
    # It takes bb (pre-trained backbone) as input, wraps the backbone as a chunk encoder and returns nothing.
    def __init__(self, bb):
        super().__init__()
        self.m, self.bb = bb.model, bb

    # It takes ids (chunk token ids) and lens (chunk lengths) as input, turns each chunk into one vector and returns one vector per chunk.
    def forward(self, ids, lens):
        sh = copy.copy(self.bb); sh.model = self.m
        h = sh.hidden(ids).float()
        mask = (torch.arange(h.size(1), device=h.device)[None] < lens[:, None]).float()
        mean = (h * mask[..., None]).sum(1) / lens[:, None].clamp(min=1)
        return torch.cat([mean, h[torch.arange(h.size(0), device=h.device), (lens - 1).clamp(min=0)]], -1)

class Head(nn.Module):
    # It takes din (decoder input), K (number of labels), prior (label frequencies), hid (hidden size), gru (use a GRU) and drop (dropout) as input, creates the head that combines chunk vectors into a prediction and returns nothing.
    def __init__(self, din, K, prior, hid=384, gru=256, drop=0.3):
        super().__init__()
        self.norm, self.drop = nn.LayerNorm(din), nn.Dropout(drop)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(din, hid), nn.GELU())
        self.pos = nn.Sequential(nn.Linear(NPOS, 32), nn.GELU())
        self.gru = nn.GRU(hid + 32, gru, batch_first=True, bidirectional=True)
        self.att, self.glob = nn.Linear(2 * gru, K), nn.Linear(4 * gru, K)
        self.w = nn.Parameter(torch.randn(K, 2 * gru) * 0.02)
        self.skip = nn.Linear(din, K)
        p = torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 0.5)
        self.glob.bias.data = torch.log(p / (1 - p))

    # It takes x (input tensor), pos (chunk positions) and cm (chunk mask) as input, combines chunk vectors and returns the prediction scores.
    def forward(self, x, pos, cm):
        xn = self.norm(x.float())
        z = torch.cat([self.proj(xn), self.pos(pos)], -1)
        y, _ = self.gru(nn.utils.rnn.pack_padded_sequence(z, cm.sum(1).clamp(min=1).cpu(), batch_first=True, enforce_sorted=False))
        y = self.drop(nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=cm.size(1))[0])
        a = self.att(y).masked_fill(~cm[..., None], -1e4).softmax(1)
        mean = (y * cm[..., None]).sum(1) / cm.sum(1, keepdim=True).clamp(min=1)
        return (torch.einsum("bmk,bmd->bkd", a, y) * self.w).sum(-1) + self.glob(torch.cat([mean, y.masked_fill(~cm[..., None], -1e4).max(1)[0]], -1)) + self.skip(self.drop(xn.masked_fill(~cm[..., None], -1e4).max(1)[0]))

class DocModel(nn.Module):
    # It takes bb (pre-trained backbone), cfg (settings), K (number of labels) and prior (label frequencies) as input, joins the chunk encoder and the head into one model and returns nothing.
    def __init__(self, bb, cfg, K, prior):
        super().__init__()
        self.enc, self.head, self.dp = Enc(bb), Head(2 * bb.d_model, K, prior, cfg["hid"], cfg["gru"], cfg["head_dropout"]), None

    # It takes ids (chunk token ids) and lens (chunk lengths) as input, gets chunk vectors and returns chunk vectors.
    def vec(self, ids, lens):
        return (self.dp or self.enc)(ids, lens)

    # It takes b (batch dictionary) as input, runs the whole model on a batch of documents and returns prediction scores.
    def forward(self, b):
        v = self.vec(b["ids"], b["lens"])
        x = v.new_zeros(b["cm"].shape[0], b["cm"].shape[1], v.size(-1))
        x[b["bi"], b["ki"]] = v
        return self.head(x, b["pos"], b["cm"])

class Paused(Exception):
    pass

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def is_oom(e):
    return "out of memory" in str(e).lower()

# It takes model, docs (documents), dev (device), pad (padding id), take (chunk chooser) and cb (chunks per batch) as input, computes chunk vectors for all documents and returns chunk vectors per document.
@torch.no_grad()
def encode(model, docs, dev, pad, take, cb=32):
    model.eval()
    items = [(di, i, len(d.chunk(i))) for di, d in enumerate(docs) for i in take(len(d.segs))]
    order = sorted(range(len(items)), key=lambda k: items[k][2])
    V, p = [dict() for _ in docs], 0
    while p < len(order):
        part = [items[k] for k in order[p:p + cb]]
        arrs = [docs[di].chunk(i) for di, i, _ in part]
        ids = np.full((len(arrs), max(map(len, arrs))), pad, np.int64); lens = np.array(list(map(len, arrs)), np.int64)
        for r, a in enumerate(arrs):
            ids[r, :len(a)] = a
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
                v = model.vec(torch.from_numpy(ids).to(dev), torch.from_numpy(lens).to(dev)).float().cpu().numpy()
        except RuntimeError as e:
            if is_oom(e) and cb > 1:
                torch.cuda.empty_cache(); cb = max(1, cb // 2); continue
            raise
        for r, (di, i, _) in enumerate(part):
            V[di][i] = v[r].astype(np.float16)
        p += len(part)
    return V

# It takes docs (documents), V (chunk vectors) and sel (chunk chooser) as input, packs chunk vectors into padded arrays and returns (vectors, positions, mask).
def pack(docs, V, sel):
    ix = [[i for i in sel(len(d.segs)) if i in V[k]] or sorted(V[k]) for k, d in enumerate(docs)]
    km, D = max(map(len, ix)), len(next(iter(V[0].values())))
    X = np.zeros((len(docs), km, D), np.float16); P = np.zeros((len(docs), km, NPOS), np.float32); M = np.zeros((len(docs), km), bool)
    for r, (d, ii) in enumerate(zip(docs, ix)):
        for j, i in enumerate(ii):
            X[r, j], P[r, j], M[r, j] = V[r][i], d.pos[i], True
    return X, P, M

# It takes head (trained head), X (chunk vectors), P (chunk positions), M (chunk mask), dev (device) and bs (batch size) as input, gets scores from the head and returns an array of scores.
@torch.no_grad()
def head_logits(head, X, P, M, dev, bs=256):
    head.eval()
    t = lambda a, s: torch.from_numpy(a[s:s + bs]).to(dev)
    return np.concatenate([head(t(X, s), t(P, s), t(M, s)).float().cpu().numpy() for s in range(0, len(X), bs)])

# It takes k (number of chunks to pick) as input, chooses chunks spread evenly over the document and returns a function that picks chunk indices.
def spread(k):
    return lambda m: sorted({int(round(x)) for x in np.linspace(0, m - 1, min(m, k))})

MODES = {"all": lambda m: list(range(m)), "first1": lambda m: [0], "first4": lambda m: list(range(min(4, m))),
         "last4": lambda m: list(range(max(0, m - 4), m)), "spread4": spread(4)}

# It takes z (values) as input, returns values between 0 and 1; it is the sigmoid function.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# It takes Y (true labels (0/1 matrix)) and P (probabilities) as input, returns F1 table (thresholds x labels); it is the F1 of each label for many thresholds.
def f1_grid(Y, P):
    Y = Y > 0; out = []
    for t in GRID:
        pr = P >= t; tp = (pr & Y).sum(0)
        out.append(2 * tp / np.maximum(pr.sum(0) + Y.sum(0), 1))
    return np.array(out)

# It takes Y (true labels (0/1 matrix)) and P (probabilities) as input, returns the best macro F1; it is the best macro F1 with one shared threshold.
def gmacro(Y, P):
    return float(f1_grid(Y, P)[:, (Y > 0).sum(0) > 0].mean(1).max())

# It takes Y (true labels (0/1 matrix)), P (probabilities), m (smoothing strength) and smooth (smoothing width) as input, finds a threshold for each label and returns a threshold for each label.
def fit_thr(Y, P, m=8, smooth=1):
    Fg = f1_grid(Y, P); k = np.ones(2 * smooth + 1) / (2 * smooth + 1)
    Fg = np.apply_along_axis(lambda c: np.convolve(np.pad(c, smooth, mode="edge"), k, "valid"), 0, Fg)
    sup = (Y > 0).sum(0)
    g = GRID[Fg[:, sup > 0].mean(1).argmax()]
    w = sup / (sup + m)
    return w * GRID[Fg.argmax(0)] + (1 - w) * g

# It takes P (probabilities), thr (threshold) and top1 (always keep the best label) as input, turns probabilities into labels (at least one per case) and returns a 0/1 label matrix.
def decide(P, thr, top1=True):
    pr = P >= thr
    if top1:
        e = ~pr.any(1); pr[e, (P / thr)[e].argmax(1)] = True
    return pr

# It takes Y (true labels (0/1 matrix)) and pr (predicted labels) as input, computes macro F1, micro F1 and other scores and returns a dictionary of scores.
def score(Y, pr):
    Y = Y > 0; tp = (pr & Y).sum(0).astype(float); fp, fn = pr.sum(0) - tp, Y.sum(0) - tp; sup = Y.sum(0) > 0
    f = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return dict(macro_f1=float(f[sup].mean()), macro_p=float((tp / np.maximum(tp + fp, 1))[sup].mean()), macro_r=float((tp / np.maximum(tp + fn, 1))[sup].mean()),
                micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)), subset_acc=float((pr == Y).all(1).mean()), per_label_f1=f.tolist())

# It takes Y (true labels (0/1 matrix)), P (probabilities), top1 (always keep the best label), m (smoothing strength) and folds (number of folds) as input, scores thresholds with cross-validation and returns macro F1.
def cv_score(Y, P, top1=True, m=8, folds=5):
    idx = np.random.default_rng(0).permutation(len(Y)); pr = np.zeros(Y.shape, bool)
    for f in range(folds):
        te = idx[f::folds]; pr[te] = decide(P[te], fit_thr(Y[np.setdiff1d(idx, te)], P[np.setdiff1d(idx, te)], m), top1)
    return score(Y, pr)["macro_f1"]

# It takes lg (logits), y (labels), pw (positive weights) and kind (type) as input, returns a loss value; it is the multi-label loss.
def loss_fn(lg, y, pw, kind="bce"):
    if kind == "asl":
        p = lg.sigmoid(); pm = (p - 0.05).clamp(min=0)
        return -(y * F.logsigmoid(lg) + (1 - y) * pm ** 4 * torch.log((1 - pm).clamp(min=1e-6))).mean()
    return F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)

# It takes Y (true labels (0/1 matrix)), power, cap (maximum value) and dev (device) as input, gives rare labels a higher loss weight and returns a tensor of label weights.
def pos_weight(Y, power, cap, dev):
    n = Y.sum(0).clip(1)
    return torch.tensor(np.clip(((len(Y) - n) / n) ** power, 1, cap), dtype=torch.float32, device=dev)

# It takes step, warm (warm-up steps) and total (total steps) as input, returns a learning-rate multiplier; it is the learning-rate multiplier: warm-up then cosine decay.
def lr_factor(step, warm, total):
    return (step + 1) / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm)))))

# It takes tr (training data), dv (dev data), cfg (settings), prior (label frequencies), dev (device), kind (type), seed (random seed), epochs (number of epochs), lr (learning rate), bs (batch size), init (starting model) and log (print function) as input, trains the head on saved chunk vectors and returns (trained head, best dev score).
def fit_head(tr, dv, cfg, prior, dev, kind="bce", seed=0, epochs=25, lr=1e-3, bs=64, init=None, log=print):
    (X, P, M, Y), (Xd, Pd, Md, Yd) = tr, dv
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    h = init or Head(X.shape[-1], Y.shape[1], prior, cfg["hid"], cfg["gru"], cfg["head_dropout"]).to(dev)
    opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=0.01)
    pw = pos_weight(Y, cfg["pw_power"], cfg["pw_max"], dev)
    spe = math.ceil(len(X) / bs); total, best, state = epochs * spe, -1.0, None
    step = 0
    for ep in range(epochs):
        h.train(); perm = rng.permutation(len(X))
        for s in range(0, len(X), bs):
            b = perm[s:s + bs]; t = lambda a: torch.from_numpy(a[b]).to(dev)
            x, p, m = t(X), t(P), t(M)
            m = m & (torch.rand(m.shape, device=dev) > cfg["chunk_drop"]); m[:, 0] |= ~m.any(1)
            o = (~m).to(torch.uint8).argsort(dim=1, stable=True)
            x, p, m = x.gather(1, o[..., None].expand(-1, -1, x.size(-1))), p.gather(1, o[..., None].expand(-1, -1, p.size(-1))), m.gather(1, o)
            for g in opt.param_groups:
                g["lr"] = lr * lr_factor(step, 50, total)
            loss = loss_fn(h(x, p, m), t(Y), pw, kind)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(h.parameters(), 1.0); opt.step(); step += 1
        f = gmacro(Yd, sigmoid(head_logits(h, Xd, Pd, Md, dev)))
        if f > best:
            best, state = f, copy.deepcopy(h.state_dict())
    h.load_state_dict(state)
    log(f"  head fit ({kind}, seed {seed}): dev macro-F1 {best:.4f}")
    return h, best

class EMA:
    # It takes params (parameters) and decay (decay factor) as input, starts a moving-average copy of the weights and returns nothing.
    def __init__(self, params, decay):
        self.p, self.d, self.n = params, decay, 0
        self.e = [p.detach().clone().float() for p in params]

    # It takes nothing (uses the stored weights) as input, updates the moving-average weights and returns nothing.
    def update(self):
        self.n += 1
        d = min(self.d, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self.e, d)
        torch._foreach_add_(self.e, [p.detach().float() for p in self.p], alpha=1 - d)

    # It uses the moving-average weights for evaluation and returns nothing.
    def swap_in(self):
        self.bak = [p.detach().clone() for p in self.p]
        for p, e in zip(self.p, self.e):
            p.data.copy_(e.to(p.dtype))

    # It puts the normal weights back and returns nothing.
    def restore(self):
        for p, b in zip(self.p, self.bak):
            p.data.copy_(b)

# It takes s (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def hms(s):
    s = int(max(0, s)); return f"{s // 3600}h{(s % 3600) // 60:02d}m"

# It takes model and params (parameters) as input, copies the trainable weights and returns a dictionary of trainable weights.
def tstate(model, params):
    ids = {id(p) for p in params}
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if id(p) in ids}

# It takes model and st (saved weights) as input, loads saved trainable weights and returns nothing.
def load_tstate(model, st):
    sd = dict(model.named_parameters())
    for n, v in st.items():
        sd[n].data.copy_(v.to(sd[n].dtype))

# It takes model, tr (training data), dv (dev data), cfg (settings), groups (parameter groups with learning rates), unfreeze (unfreeze schedule), unfreeze_fn (function that unfreezes layers), dev (device), pad (padding id), name, work (work folder) and log (print function) as input, returns the best weights and training history; it is the main training loop with evaluation and saving.
def train(model, tr, dv, cfg, groups, unfreeze, unfreeze_fn, dev, pad, name, work, log=print):
    use_amp = dev == "cuda"
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), weight_decay=cfg["wd"])
    base = [g["lr"] * cfg.get("lr_scale", 1.0) for g in groups]
    scaler = torch.amp.GradScaler("cuda", init_scale=cfg["init_scale"], enabled=use_amp)
    ema = EMA(params, cfg["ema"])
    Ytr = np.stack([d.y for d in tr]); pw = pos_weight(Ytr, cfg["pw_power"], cfg["pw_max"], dev)
    if torch.cuda.device_count() > 1:
        object.__setattr__(model, "dp", nn.DataParallel(model.enc))
    bsz, mb = cfg["batch_docs"], cfg["mb"]
    spe = math.ceil(len(tr) / bsz); total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * total))
    ck = os.path.join(work, f"{name}.resume.pt")
    cand = [c for c in [ck] + glob.glob(f"/kaggle/input/**/{name}.resume.pt", recursive=True) + glob.glob(f"/kaggle/input/**/{name}.resume.pt/**/*", recursive=True) if os.path.isfile(c)]
    step, best, best_state, hist, used, cal_k = 0, -1.0, None, [], 0.0, -2
    if cand:
        try:
            c = torch.load(cand[0], map_location="cpu", weights_only=False)
            load_tstate(model, c["state"]); opt.load_state_dict(c["opt"]); scaler.load_state_dict(c["scaler"])
            for e, s in zip(ema.e, c["ema"]):
                e.copy_(s)
            ema.n, step, best, best_state, hist, total, used = c["ema_n"], c["step"], c["best"], c["best_state"], c["hist"], c["total"], c.get("used", 0.0)
            log(f"resumed at step {step}/{total}")
        except Exception as e:
            log(f"resume failed: {str(e)[:100]}"); step, best, best_state, hist, used = 0, -1.0, None, [], 0.0
    ev_docs = dv
    if len(dv) > cfg["eval_docs"]:
        ev_docs = [dv[i] for i in sorted(np.random.default_rng(0).choice(len(dv), cfg["eval_docs"], replace=False))]
    Yev = np.stack([d.y for d in ev_docs])
    max_s, sess_s = cfg["max_hours"] * 3600, cfg.get("session_hours", cfg["max_hours"]) * 3600
    cur_k, nskip, lr_mult, rollbacks, bad_n = -1, 0, 1.0, 0, 0
    t_sess, step0, t_save, t_ev, pstd = time.time(), step, time.time(), 0.0, 0.0
    stage_t, stage_s, stage_ev, gn_sum, gn_n, loss_sum, loss_n, recent = t_sess, step, 0.0, 0.0, 0, 0.0, 0, []
    name_of = {id(p): n for n, p in model.named_parameters()}
    ev_every = max(1, total // cfg["evals"])

    # It uses values from the outer function, quickly scores the model on dev documents and returns the dev score and current weights.
    def quick_eval():
        nonlocal t_ev, pstd
        t = time.time(); ema.swap_in()
        try:
            V = encode(model, ev_docs, dev, pad, spread(cfg["eval_ch"]), cfg["cb"])
            p = sigmoid(head_logits(model.head, *pack(ev_docs, V, MODES["all"]), dev)); pstd = float(p.std(0).mean())
            f, st = gmacro(Yev, p), tstate(model, params)
        finally:
            ema.restore()
        t_ev += time.time() - t
        return f, st

    model.train()
    if cfg.get("continue") and best_state is None:
        model.eval(); best, best_state = quick_eval(); model.train(); log(f"  starting point (loaded weights): dev macro-F1 {best:.4f}")
    while step < total:
        ep = step // spe
        perm = np.random.default_rng(cfg["seed"] * 100003 + ep).permutation(len(tr))
        try:
            from tqdm.auto import tqdm
            bar = tqdm(total=total, initial=step, desc="train", mininterval=60, dynamic_ncols=True)
        except Exception:
            bar = None
        for bi in range(step - ep * spe, spe):
            if step >= total:
                break
            if unfreeze:
                k = [kk for f, kk in unfreeze if step / max(1, total) >= f][-1]
                if k != cur_k:
                    unfreeze_fn(k); cur_k = k; stage_t, stage_s, stage_ev = time.time(), step, t_ev; log(f"  unfrozen top {k} blocks at step {step}")
            lr_f = (lr_factor(step, warm, total) * min(1.0, 0.05 + (step - stage_s + 1) / cfg["ramp_steps"]) if step > warm else lr_factor(step, warm, total)) * lr_mult
            for g, b0 in zip(opt.param_groups, base):
                g["lr"] = b0 * lr_f
            rng = np.random.default_rng((cfg["seed"] * 100003 + ep) * 7919 + bi)
            items = []
            for di in perm[bi * bsz:(bi + 1) * bsz]:
                d = tr[di]; m = len(d.segs); k = int(rng.integers(cfg["k_lo"], cfg["k_hi"] + 1))
                items.append((d, list(range(m)) if m <= k else sorted(rng.choice(m, k, replace=False).tolist())))
            y_all = torch.tensor(np.stack([d.y for d, _ in items]), device=dev)
            z_all = torch.tensor(np.stack([d.soft for d, _ in items]), device=dev) if cfg["kd_alpha"] > 0 else None
            while True:
                try:
                    run = 0.0
                    for s in range(0, len(items), mb):
                        b = collate(items[s:s + mb], pad, dev, cfg["tok_mask"], rng)
                        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                            lg = model(b).float()
                        ce = loss_fn(lg, y_all[s:s + mb], pw, cfg["loss"])
                        if z_all is not None:
                            T = cfg["kd_T"]
                            kd = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(z_all[s:s + mb] / T)) * T * T
                            ce = (1 - cfg["kd_alpha"]) * ce + cfg["kd_alpha"] * kd
                        loss = ce * len(items[s:s + mb]) / len(items)
                        scaler.scale(loss).backward(); run += float(loss.detach())
                    break
                except RuntimeError as e:
                    if is_oom(e) and mb > 1:
                        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); mb = max(1, mb // 2); log(f"  out of memory -> {mb} documents per micro-batch")
                        continue
                    raise
            scaler.unscale_(opt); gn = float(nn.utils.clip_grad_norm_([p for p in params if p.grad is not None], cfg["clip"]))
            if math.isfinite(gn):
                gn_sum += gn; gn_n += 1
            loss_sum += run; loss_n += 1
            sc0 = scaler.get_scale()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(); step += 1
            nskip += scaler.get_scale() < sc0; recent = (recent + [scaler.get_scale() < sc0])[-100:]
            if step == cfg["sanity_step"] and np.mean(recent) > 0.4:
                raise RuntimeError("most optimizer steps were skipped (inf/NaN gradients): training is not updating the weights")
            if bar is not None:
                bar.update(1)
            el = used + time.time() - T0
            if cal_k != cur_k and step - stage_s >= cfg["cal_steps"]:
                cal_k = cur_k
                sps = (time.time() - stage_t - (t_ev - stage_ev)) / (step - stage_s)
                allow = (cfg["train_frac"] * max_s - el) / (sps * 1.1)
                if step + allow < total:
                    total = max(warm + 50, int(step + allow)); ev_every = max(1, total // cfg["evals"]); log(f"  time budget: {sps:.2f}s/step -> total steps reduced to {total}")
                else:
                    log(f"  {sps:.2f}s/step, projected training time {hms((total - step) * sps * 1.1)}")
            stop = el > cfg["stop_frac"] * max_s
            pause = (time.time() - T0 > 0.93 * sess_s) and step < total and not stop
            if step % ev_every == 0 or step >= total or stop:
                model.eval(); f, st = quick_eval(); model.train()
                if f > best:
                    best, best_state = f, st
                tl, gnm = loss_sum / max(1, loss_n), gn_sum / max(1, gn_n); loss_sum, loss_n, gn_sum, gn_n = 0.0, 0, 0.0, 0
                hist.append(dict(step=step, epoch=step / spe, dev_macro_f1=f, loss=tl, grad_norm=gnm, lr_mult=lr_mult, skipped=int(nskip), prob_std=pstd))
                log(f"  step {step}/{total} (epoch {step / spe:.2f}) dev macro-F1 {f:.4f} best {best:.4f} | loss {tl:.4f} grad {gnm:.2f} lr x{lr_mult:g} | prob std {pstd:.4f} skipped {int(nskip)} | ETA {hms((total - step) * (time.time() - t_sess - t_ev) / max(1, step - step0))}")
                if f < best - cfg["drop_tol"] and best_state is not None and step < total:
                    if rollbacks < cfg["max_rollbacks"]:
                        load_tstate(model, best_state)
                        for p, e in zip(params, ema.e):
                            e.copy_(best_state[name_of[id(p)]].float())
                        opt.state.clear(); scaler.update(4096.0 if use_amp else 1.0); lr_mult *= 0.5; rollbacks += 1; stage_s = step
                        log(f"  rolled back to the best weights (dev {best:.4f}); learning rates now x{lr_mult:g} (rollback {rollbacks})")
                    else:
                        log("  dev score keeps falling: stopping and keeping the best weights"); total = step
                elif len(hist) >= cfg["patience"] and max(h["dev_macro_f1"] for h in hist[-cfg["patience"]:]) <= best - 1e-9 and step > 0.6 * total:
                    log("  no dev improvement for several evaluations: stopping"); total = step
            if (time.time() - t_save > cfg["save_every"] and step < total) or (stop and step < total) or pause:
                torch.save(dict(state=tstate(model, params), opt=opt.state_dict(), scaler=scaler.state_dict(), ema=[e.cpu() for e in ema.e], ema_n=ema.n,
                                step=step, best=best, best_state=best_state, hist=hist, total=total, used=el), ck + ".tmp")
                os.replace(ck + ".tmp", ck); t_save = time.time()
            if pause:
                raise Paused(f"session time is nearly over: progress saved at step {step}/{total}. Run this notebook again with this run's output attached as an input to continue")
            if stop:
                log("  time limit reached: stopping training"); total = step
                break
        if bar is not None:
            bar.close()
    if best_state is None:
        ema.swap_in(); best_state = tstate(model, params); ema.restore()
    if os.path.exists(ck):
        os.remove(ck)
    return best_state, hist, params

# It takes path (file path) and preds (predictions) as input, writes predictions into a submission zip and returns nothing.
def write_zip(path, preds):
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(preds))
    with zipfile.ZipFile(path) as z:
        assert z.namelist() == ["prediction.json"]

# It takes ids (token ids), pr (predicted labels) and labels (label names) as input, formats predictions for submission and returns a list of predictions in submission format.
def make_preds(ids, pr, labels):
    assert len(ids) == len(set(map(str, ids))) == len(pr)
    return [{"id": i, "pred": [int(labels[j]) for j in np.flatnonzero(r)]} for i, r in zip(ids, pr)]

# It takes raw (raw documents) and index (label to column map) as input, turns label lists into 0/1 vectors and returns a 0/1 label matrix.
def multihot(raw, index):
    Y = np.zeros((len(raw), len(index)), np.float32)
    for r, it in enumerate(raw):
        for l in it["y"] or []:
            if l in index:
                Y[r, index[l]] = 1
    return Y

# It takes a (first set of arrays) and b (second set of arrays) as input, joins two sets of arrays together and returns joined arrays.
def _cat(a, b):
    km = max(a[0].shape[1], b[0].shape[1])
    pd = lambda x: x if x.shape[1] == km else np.pad(x, [(0, 0), (0, km - x.shape[1])] + [(0, 0)] * (x.ndim - 2))
    return tuple(np.concatenate([pd(x), pd(y)]) for x, y in zip(a, b))

# It takes enc (encoded document), Yt (train labels), Yd (dev labels), cfg (settings), prior (label frequencies), dev (device) and log (print function) as input, trains heads on folds of dev data and averages them and returns a dictionary with dev and test probabilities.
def cv_refit(enc, Yt, Yd, cfg, prior, dev, log=print):
    folds = cfg.get("refit_folds", 4); idx = np.random.default_rng(0).permutation(len(Yd))
    D, T = np.zeros(Yd.shape, np.float32), np.zeros((len(enc["test"][0]), Yd.shape[1]), np.float32)
    runs = [(kd, s) for kd in cfg["refit_losses"] for s in range(cfg["refit_seeds"])]
    for f in range(folds):
        te = np.sort(idx[f::folds]); tr = np.setdiff1d(idx, te)
        dte = tuple(a[te] for a in enc["dev"]); full = (*_cat(enc["train"], tuple(a[tr] for a in enc["dev"])), np.concatenate([Yt, Yd[tr]]))
        for kd, s in runs:
            h = fit_head(full, (*dte, Yd[te]), cfg, prior, dev, kind=kd, seed=s + 10 * f, epochs=cfg["refit_epochs"], log=log)[0]
            D[te] += head_logits(h, *dte, dev) / len(runs); T += head_logits(h, *enc["test"], dev) / (len(runs) * folds)
        del full
    return {"dev": D, "test": T}

# It takes name, A (model code module), ckpt (checkpoint path), tokf (tokenizer file path), build (function that builds the model), cfg (settings), out (output folder), work (work folder), teacher_dir (teacher output folder) and log (print function) as input, runs the whole pipeline: data, training, evaluation and submissions and returns a results summary.
def main(name, A, ckpt, tokf, build, cfg, out, work, teacher_dir=None, log=print):
    os.makedirs(out, exist_ok=True); os.makedirs(work, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); random.seed(cfg["seed"])
    paths = ensure_data()
    raw = {k: load_docs(paths[k + ".json"]) for k in ("train", "dev", "test")}
    if cfg["smoke"]:
        raw = {k: v[:cfg["smoke"]] for k, v in raw.items()}
    th = lambda r: hashlib.md5(" ".join(str(r["text"]).split()).encode()).hexdigest()
    tid, tx = {str(r["id"]) for r in raw["test"]}, {th(r) for r in raw["test"]}
    log(f"overlap with test: dev by id {sum(str(r['id']) in tid for r in raw['dev'])}, dev by text {sum(th(r) in tx for r in raw['dev'])}, train by id {sum(str(r['id']) in tid for r in raw['train'])}, train by text {sum(th(r) in tx for r in raw['train'])}")
    keep = [r for r in raw["dev"] if str(r["id"]) not in tid and th(r) not in tx]
    if len(keep) != len(raw["dev"]):
        log(f"removed {len(raw['dev']) - len(keep)} dev documents that are also test cases"); raw["dev"] = keep
    labels = sorted({l for r in raw["train"] for l in r["y"]}); index = {l: j for j, l in enumerate(labels)}; K = len(labels)
    unseen = sum(l not in index for r in raw["dev"] for l in r["y"])
    log({k: len(v) for k, v in raw.items()}, f"{K} statute labels in train; dev labels not in train: {unseen}; mean labels per train doc {np.mean([len(r['y']) for r in raw['train']]):.2f}")
    tok = A.load_tokenizer(tokf); pad = tok.pad_id
    docs = {k: tokenize_docs(raw[k], tok, cfg, os.path.join(work, f"{name}_{k}")) for k in raw}
    Y = {k: multihot(raw[k], index) for k in ("train", "dev")}
    for k in Y:
        for d, y in zip(docs[k], Y[k]):
            d.y = y
    prior = Y["train"].mean(0)
    bb, groups_fn, uf = build()
    model = DocModel(bb, cfg, K, prior).to(dev)
    groups = groups_fn(model)
    sched, ufn = uf if uf else (None, None)
    ntr, nall = sum(p.numel() for g in groups for p in g["params"]), sum(p.numel() for p in model.parameters())
    log(f"{name}: trainable parameters {ntr:,} of {nall:,}")
    if teacher_dir:
        fs = glob.glob(os.path.join(teacher_dir, "**", "*_train_logits.npz"), recursive=True)
        if not fs:
            for z in glob.glob(os.path.join(teacher_dir, "**", "*_ALL_outputs.zip"), recursive=True):
                zipfile.ZipFile(z).extractall(os.path.join(work, "teacher"))
            fs = glob.glob(os.path.join(work, "teacher", "**", "*_train_logits.npz"), recursive=True)
        assert fs, "teacher logits (*_train_logits.npz) not found"
        t = np.load(fs[0], allow_pickle=True); log(f"teacher file {fs[0]} ({len(fs)} found)")
        assert [int(x) for x in t["labels"]] == labels, "teacher label set differs from this run"
        row = {str(i): r for r, i in enumerate(t["ids"])}
        for d in docs["train"]:
            d.soft = t["z"][row[str(d.id)]].astype(np.float32)
    if cfg["lp_docs"]:
        sub = [docs["train"][i] for i in np.random.default_rng(1).permutation(len(docs["train"]))[:cfg["lp_docs"]]]
        log(f"linear-probe start: training the head on {len(sub)} documents with the frozen backbone")
        Vs, Vd = encode(model, sub, dev, pad, spread(cfg["k_hi"]), cfg["cb"]), encode(model, docs["dev"][:cfg["eval_docs"]], dev, pad, spread(cfg["k_hi"]), cfg["cb"])
        model.head, _ = fit_head((*pack(sub, Vs, MODES["all"]), np.stack([d.y for d in sub])), (*pack(docs["dev"][:cfg["eval_docs"]], Vd, MODES["all"]), Y["dev"][:cfg["eval_docs"]]),
                                 cfg, prior, dev, init=model.head, epochs=cfg["lp_epochs"], log=log)
        del Vs, Vd
    t_train = time.time()
    sk = glob.glob(f"/kaggle/input/**/{cfg.get('init_name', name)}_best_trainable.pt", recursive=True)
    if cfg["skip_train"] and sk:
        best_state, hist, params = torch.load(sk[0], map_location="cpu"), [], []
        load_tstate(model, best_state); log(f"skipping training: loaded {sk[0]}")
    else:
        if cfg.get("continue") and sk:
            load_tstate(model, torch.load(sk[0], map_location="cpu")); log(f"continuing from {sk[0]}")
        try:
            best_state, hist, params = train(model, docs["train"], docs["dev"], cfg, groups, sched, ufn, dev, pad, name, work, log)
        except Paused as e:
            log(str(e)); return None
        torch.save(best_state, os.path.join(out, f"{name}_best_trainable.pt"))
        load_tstate(model, best_state)
    train_secs = time.time() - t_train
    summary = dict(name=name, labels=labels, cfg={k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool, list, tuple, type(None)))}, trainable_params=ntr, total_params=nall,
                   backbone_params=int(bb.n_params), model_mb_fp32=nall * 4 / 2**20, train_seconds=train_secs, history=hist,
                   peak_gpu_gb=torch.cuda.max_memory_allocated() / 2**30 if dev == "cuda" else 0.0)
    enc = {}
    for k in ("train", "dev", "test"):
        t = time.time(); enc[k] = pack(docs[k], encode(model, docs[k], dev, pad, MODES["all"], cfg["cb"]), MODES["all"])
        summary[f"encode_seconds_per_doc_{k}"] = (time.time() - t) / len(docs[k])
    Yt, Yd = Y["train"], Y["dev"]
    logits = {"ft": {k: head_logits(model.head, *enc[k], dev) for k in ("train", "dev", "test")}}
    dvp = logits["ft"]["dev"]
    log(f"[ft] dev macro-F1 with one global threshold {gmacro(Yd, sigmoid(dvp)):.4f}")
    summary["chunk_mode_ablation"] = {}
    del model, bb
    torch.cuda.empty_cache()
    ref = cv_refit(enc, Yt, Yd, cfg, prior, dev, log)
    logits["refit"] = ref; logits["blend"] = {k: (logits["ft"][k] + ref[k]) / 2 for k in ("dev", "test")}
    dev_ids, test_ids = [d.id for d in docs["dev"]], [d.id for d in docs["test"]]
    for cand, L in logits.items():
        Pd, Pt = sigmoid(L["dev"]), sigmoid(L["test"])
        cvs = {t1: cv_score(Yd, Pd, t1) for t1 in (True, False)}; top1 = max(cvs, key=cvs.get)
        thr = fit_thr(Yd, Pd); m = score(Yd, decide(Pd, thr, top1))
        summary[cand] = dict(top1=top1, cv_macro_f1=cvs[top1], cv_without_top1=cvs[not top1], dev_in_sample={k: v for k, v in m.items() if k != "per_label_f1"},
                             global_thr_macro_f1=gmacro(Yd, Pd), at_0_5=score(Yd, Pd >= 0.5)["macro_f1"], per_label_thr_macro_f1=score(Yd, decide(Pd, thr, False))["macro_f1"])
        log(f"[{cand}] dev macro-F1: cv {cvs[top1]:.4f} | in-sample {m['macro_f1']:.4f} (P {m['macro_p']:.3f} R {m['macro_r']:.3f}) micro {m['micro_f1']:.4f} subset {m['subset_acc']:.4f} | @0.5 {summary[cand]['at_0_5']:.4f} | global thr {summary[cand]['global_thr_macro_f1']:.4f} | top-1 rule {top1}")
        np.savez_compressed(os.path.join(out, f"{name}_preds_{cand}.npz"), dev_ids=np.array(dev_ids, dtype=str), dev_p=Pd.astype(np.float16), dev_y=Yd.astype(bool),
                            test_ids=np.array(test_ids, dtype=str), test_ids_json=np.array(json.dumps(test_ids)), test_p=Pt.astype(np.float16), labels=np.array(labels))
        write_zip(os.path.join(out, f"{name}_{cand}_submission.zip"), make_preds(test_ids, decide(Pt, thr, top1), labels))
        if cand == "ft":
            pr = decide(Pd, thr, top1); f1 = np.array(m["per_label_f1"]); ntr_l = Yt.sum(0)
            summary["per_label"] = [dict(label=labels[j], train=int(ntr_l[j]), dev=int(Yd[:, j].sum()), f1=float(f1[j])) for j in range(K)]
            q = np.quantile(ntr_l, [1 / 3, 2 / 3])
            summary["macro_f1_by_train_frequency"] = {nm: float(f1[(Yd.sum(0) > 0) & s].mean()) for nm, s in (("rare", ntr_l <= q[0]), ("mid", (ntr_l > q[0]) & (ntr_l <= q[1])), ("frequent", ntr_l > q[1]))}
            miss, fp = (Yd > 0) & ~pr, pr & (Yd == 0); pairs = {}
            for i in range(len(Yd)):
                for a in np.flatnonzero(miss[i]):
                    for b in np.flatnonzero(fp[i]):
                        pairs[(labels[a], labels[b])] = pairs.get((labels[a], labels[b]), 0) + 1
            summary["most_confused_missed_vs_predicted"] = [dict(missed=a, predicted_instead=b, docs=c) for (a, b), c in sorted(pairs.items(), key=lambda x: -x[1])[:15]]
            log("macro-F1 by training frequency:", summary["macro_f1_by_train_frequency"])
    if cfg["dump_train"]:
        np.savez_compressed(os.path.join(out, f"{name}_train_logits.npz"), ids=np.array([d.id for d in docs["train"]], dtype=str), z=logits["ft"]["train"].astype(np.float16), labels=np.array(labels))
    json.dump(summary, open(os.path.join(out, f"{name}_summary.json"), "w"), indent=1, default=float)
    for f in glob.glob(os.path.join(work, "*.pkl")):
        os.remove(f)
    zp = os.path.join(out, f"{name}_ALL_outputs.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(os.path.join(out, "*"))):
            if os.path.isfile(f) and not f.endswith("_ALL_outputs.zip") and "phase5_data" not in f:
                z.write(f, os.path.basename(f))
    log(zp, f"{os.path.getsize(zp) / 1e6:.1f} MB", "total time", hms(time.time() - T0))
    return summary

# It takes model, head_lr (head learning rate), peft_lr (learning rate for the adapter weights) and wd (weight decay) as input, sets learning rates for the head and the backbone and returns a list of parameter groups with learning rates.
def default_groups(model, head_lr, peft_lr, wd=0.01):
    bbp = [p for p in model.enc.parameters() if p.requires_grad]
    g = [dict(params=list(model.head.parameters()), lr=head_lr, weight_decay=wd)]
    return g + [dict(params=bbp, lr=peft_lr, weight_decay=wd)] if bbp else g

Writing lsi_core.py


In [3]:
%%writefile lsi_peft.py
# Fine-tuning methods: LoRMA, layer-wise learning rates (saved as lsi_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    return list(bb.block_modules()) if hasattr(bb, "block_modules") else list(bb.model.blocks)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

class LoRMALinear(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes bb (pre-trained backbone), names (layer names), r (rank) and alpha (scale of the low-rank update) as input, freezes the model and adds LoRMA to the chosen linear layers and returns a number of layers wrapped.
def apply_lorma(bb, names, r=8, alpha=16):
    freeze(bb.model)
    n = 0
    for name in names:
        parent = bb.model
        *path, leaf = name.split(".")
        for p in path:
            parent = getattr(parent, p)
        base = getattr(parent, leaf)
        if isinstance(base, nn.Linear):
            setattr(parent, leaf, LoRMALinear(base, r, alpha))
            n += 1
    assert n > 0, "no Linear layers wrapped"
    return n

# It takes name as input, checks whether a layer is the final norm layer and returns True or False.
def is_final_norm(name):
    return "norm_f" in name or name.startswith("norm.") or name.startswith("ln_f")

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n, seen, groups = len(blocks), set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        seen.update(id(p) for p in ps)
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

# It takes bb (pre-trained backbone), top_k (top-k), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, trains only the top layers, with smaller learning rates lower down and returns a list of parameter groups with learning rates.
def partial_groups(bb, top_k, lr_top, decay, head_params, head_lr, wd=0.01):
    freeze(bb.model)
    groups = [dict(params=list(blk.parameters()), lr=lr_top * decay ** i, weight_decay=wd) for i, blk in enumerate(reversed(get_blocks(bb)[-top_k:]))]
    norm = [p for n, p in bb.model.named_parameters() if is_final_norm(n)]
    for p in norm:
        p.requires_grad_(True)
    if norm:
        groups.append(dict(params=norm, lr=lr_top, weight_decay=0.0))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return groups

# It takes bb (pre-trained backbone) and k (number of top blocks) as input, unfreezes only the top k blocks and returns nothing.
def set_unfrozen(bb, k):
    for j, blk in enumerate(reversed(get_blocks(bb))):
        for p in blk.parameters():
            p.requires_grad_(j < k)

class CkptBlock(nn.Module):
    # It takes blk (model block) as input, wraps a block to save GPU memory and returns nothing.
    def __init__(self, blk):
        super().__init__()
        self.blk = blk

    # It returns the attention layer of the wrapped block.
    @property
    def attn(self):
        return self.blk.attn

    # It takes *a (any arguments) as input, runs the block while saving memory and returns block output.
    def forward(self, *a):
        if self.training and torch.is_grad_enabled():
            return torch.utils.checkpoint.checkpoint(self.blk, *a, use_reentrant=False)
        return self.blk(*a)

# It takes bb (pre-trained backbone) as input, wraps all blocks to save GPU memory and returns nothing.
def checkpoint_blocks(bb):
    for i, blk in enumerate(bb.model.blocks):
        bb.model.blocks[i] = CkptBlock(blk)

Writing lsi_peft.py


In [4]:
%%writefile arch_legal_transformer.py
# Legal Transformer model code (saved as arch_legal_transformer.py)
import json
import math
import os
import re as _re
import shutil
import tempfile
import time as _time
import zipfile
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "legal_transformer"
TRAINED_SEQ_TOKENS = 640
NOTEBOOK_ROPE_LEN = TRAINED_SEQ_TOKENS + 8

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():

    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}

_PRETOK_RE = _re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):

        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())

            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):

        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):

        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):

        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

# It takes head_dim (size of one head), max_len (max length), device and base (rotary base value) as input, pre-computes the sine and cosine tables for rotary positions and returns (cos table, sin table).
def build_rope_cache(head_dim, max_len, device, base=10000.0):

    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

# It takes x (input tensor) as input, rotates half of the vector, used by rotary positions and returns rotated tensor.
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

# It takes q (queries), k (keys), cos (rotary cos table), sin (rotary sin table) and pos_offset (position offset) as input, adds rotary position information to queries and keys and returns (rotated queries, rotated keys).
def apply_rope(q, k, cos, sin, pos_offset):

    T = q.size(2)
    cos_t = cos[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    sin_t = sin[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    q_rot = q * cos_t + rotate_half(q) * sin_t
    k_rot = k * cos_t + rotate_half(k) * sin_t
    return q_rot, k_rot

USE_SDPA = True

class CausalSelfAttention(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates the attention layer and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
        self.attn_drop_p = dropout
        self.resid_drop = nn.Dropout(dropout)

        self.bidirectional = False
        self.key_pad_mask = None

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs attention where each token looks only at earlier tokens and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin, pos_offset)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        Tq, Tk = q.size(2), k.size(2)
        q_idx = torch.arange(Tk - Tq, Tk, device=x.device)
        k_idx = torch.arange(Tk, device=x.device)
        causal_mask = k_idx.unsqueeze(0) > q_idx.unsqueeze(1)
        if self.bidirectional:
            causal_mask = torch.zeros_like(causal_mask)

        if USE_SDPA:
            allowed = ~causal_mask
            if self.key_pad_mask is not None:
                allowed = allowed[None, None] & self.key_pad_mask[:, None, None, :]
                allowed = allowed | (k_idx.unsqueeze(0) == q_idx.unsqueeze(1))[None, None]
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.attn_drop_p if self.training else 0.0)
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            att = att.masked_fill(causal_mask, float("-inf"))
            if self.key_pad_mask is not None:
                att = att.masked_fill(~self.key_pad_mask[:, None, None, :], -1e9)
            att = F.softmax(att, dim=-1)
            att = F.dropout(att, p=self.attn_drop_p, training=self.training)
            out = att @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.wo(out)), new_cache

class TransformerMLP(nn.Module):

    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mult)
        self.fc2 = nn.Linear(dim * mult, dim)
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = TransformerMLP(dim, dropout=dropout)

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs one Transformer block and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        a, new_cache = self.attn(self.ln1(x), cos, sin, pos_offset, cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, new_cache

class LegalTransformerLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), n_heads (number of heads), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full Legal Transformer model and returns nothing.
    def __init__(self, vocab_size, dim=896, n_layers=24, n_heads=14, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads, dropout) for _ in range(n_layers)])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:

            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(self.head_dim, max_len, device=torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)

        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            nn.init.normal_(blk.attn.wo.weight, mean=0.0, std=0.02 * residual_scale)
            nn.init.normal_(blk.mlp.fc2.weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), kv_cache (saved keys and values) and pos_offset (position offset) as input, runs the model and returns next-token scores and the cache.
    def forward(self, input_ids, kv_cache=None, pos_offset=0):

        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, lc = blk(x, self.rope_cos, self.rope_sin, pos_offset, layer_cache)
            new_cache.append(lc)
        logits = self.lm_head(self.norm_f(x))
        return logits, new_cache

class LegalTokenizer(LegalByteBPETokenizer):

    # It returns the vocabulary size.
    @property
    def vocab_size(self):
        return len(self.vocab)

    # It returns the padding token id.
    @property
    def pad_id(self):
        return self.pad_token_id

    # It returns the end-of-text token id.
    @property
    def eos_id(self):
        return self.eos_token_id

    bos_id = None

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):

        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            chunk = m.group()
            base = m.start()
            b2c = []
            for ci, ch in enumerate(chunk):
                b2c.extend([base + ci] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[b] for b in chunk.encode("utf-8"))
            pos = 0
            for piece in self._bpe(sym):
                nb = len(piece)
                ids.append(self.vocab[piece])
                offs.append((b2c[pos], b2c[pos + nb - 1] + 1))
                pos += nb
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return LegalTokenizer.load(tok_path)

# It takes path (file path) as input, finds the real checkpoint file to load and returns (file path, temporary folder or None).
def _resolve_ckpt(path):

    if os.path.isfile(path):
        return path, None
    if not os.path.isdir(path):
        raise FileNotFoundError(path)
    root = None
    for dp, _dn, fn in os.walk(path):
        if "data.pkl" in fn:
            root = dp
            break
    if root is None:
        raise FileNotFoundError(f"{path} is a folder without data.pkl")
    prefix = os.path.splitext(os.path.basename(os.path.normpath(root)))[0] or "archive"
    tmpdir = tempfile.mkdtemp(prefix="legal_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for dp, _dn, fn in os.walk(root):
            for f in sorted(fn):
                full = os.path.join(dp, f)
                z.write(full, arcname=prefix + "/" + os.path.relpath(full, root).replace(os.sep, "/"))
    return out, tmpdir

class Backbone:

    # It takes model, cfg (settings) and rope_len (rotary table length) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg, rope_len):
        self.model = model
        self.config = dict(cfg)
        self.d_model = int(cfg["dim"])
        self.max_len = TRAINED_SEQ_TOKENS
        self.rope_len = rope_len
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False
        self.causal = True

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag: bool):

        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional
        for blk in self.model.blocks:
            blk.attn.bidirectional = self.bidirectional

    # It takes input_ids (token ids) and attention_mask (mask of real tokens) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None):

        m = self.model
        B, T = input_ids.shape
        if T > self.rope_len:
            raise ValueError(f"sequence length {T} > RoPE table {self.rope_len}; "
                             f"load_backbone(..., max_len>={T}) (trained only up to {self.max_len})")
        kpm = None
        if attention_mask is not None:
            am = attention_mask.to(input_ids.device).bool()
            if not bool(am.all()):
                kpm = am
        attns = [blk.attn for blk in m.blocks]
        for a in attns:
            a.key_pad_mask = kpm
        try:
            x = m.drop(m.tok_emb(input_ids))
            for blk in m.blocks:
                x, _ = blk(x, m.rope_cos, m.rope_sin, 0, None)
            return m.norm_f(x)
        finally:
            for a in attns:
                a.key_pad_mask = None

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):

        return [n for n, mod in self.model.named_modules()
                if n.startswith("blocks.") and isinstance(mod, nn.Linear)]

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

# It takes ckpt_path (checkpoint path), device, max_len (max length) and config (model settings) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", max_len=None, config=None):
    path, tmp = _resolve_ckpt(ckpt_path)
    try:
        pkg = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    if isinstance(pkg, dict) and "state_dict" in pkg:
        sd, cfg = pkg["state_dict"], dict(pkg.get("config") or {})
    else:
        sd, cfg = pkg, {}
    if config:
        cfg.update(config)
    sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
    if "vocab_size" not in cfg or "dim" not in cfg:
        w = sd["tok_emb.weight"]
        cfg.setdefault("vocab_size", w.shape[0]); cfg.setdefault("dim", w.shape[1])
        cfg.setdefault("n_layers", 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")))
    for k in ("n_heads", "n_layers"):
        if k not in cfg:
            raise ValueError(f"checkpoint config lacks {k!r}; pass config={{'{k}': ...}}")
    cfg.setdefault("dropout", 0.1)
    cfg.setdefault("tied", "lm_head.weight" not in sd or sd["lm_head.weight"].data_ptr() == sd["tok_emb.weight"].data_ptr())
    rope_len = max(int(cfg.get("max_len", NOTEBOOK_ROPE_LEN)), int(max_len or 0))
    model = LegalTransformerLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                               n_heads=cfg["n_heads"], max_len=rope_len,
                               dropout=cfg["dropout"], tie=cfg["tied"])
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if cfg["tied"]:
        missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        print("!" * 70, f"\n!!! STATE_DICT MISMATCH  missing={missing[:10]} (n={len(missing)})  "
              f"unexpected={unexpected[:10]} (n={len(unexpected)})\n" + "!" * 70)
        raise RuntimeError("checkpoint does not match Legal-Transformer architecture")
    if cfg["tied"]:
        assert model.lm_head.weight is model.tok_emb.weight
    model.to(device)
    bb = Backbone(model, cfg, rope_len)
    bb.max_len = min(TRAINED_SEQ_TOKENS, rope_len)
    return bb

Writing arch_legal_transformer.py


In [5]:
# Build the model with LoRMA and train it
import sys, os
for _m in ("lsi_core", "lsi_peft", "arch_legal_transformer"):
    sys.modules.pop(_m, None)
sys.path.insert(0, os.getcwd())
import lsi_core as C, lsi_peft as P, arch_legal_transformer as A
WORK = os.path.join(OUT, "_work")
CFG["smoke"] = SMOKE
CFG.update({'lp_docs': 0, 'refit_folds': 4, 'refit_seeds': 1, 'refit_epochs': 16, 'dump_train': True, 'continue': True, 'epochs': 1, 'lr_scale': 0.5, 'max_hours': 6.0, 'session_hours': 11.0}, init_name='p5_1_lt_lorma')

# It builds the model with the chosen fine-tuning method and its learning rates and returns (model, learning-rate groups, info).
def build():
    bb = A.load_backbone(CKPT, device="cpu")
    n = len(bb.model.blocks)
    P.apply_lorma(bb, [x for x in bb.linear_names() if int(x.split(".")[1]) >= n - TOP_K], r=R, alpha=ALPHA)
    return bb, lambda m: C.default_groups(m, HEAD_LR, PEFT_LR), None

summary = C.main(NAME, A, CKPT, TOK, build, CFG, OUT, WORK)

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

fields of the first record: ['id', 'text', 'labels']
fields of the first record: ['id', 'text', 'labels']
fields of the first record: ['id', 'text']
overlap with test: dev by id 0, dev by text 96, train by id 0, train by text 253
removed 96 dev documents that are also test cases
{'train': 42750, 'dev': 10085, 'test': 13019} 100 statute labels in train; dev labels not in train: 0; mean labels per train doc 3.70
tokenised 42750 documents in 50s, mean 1903 tokens
tokenised 10085 documents in 15s, mean 2146 tokens
tokenised 13019 documents in 18s, mean 2075 tokens
p5_1_lt_lorma_opt: trainable parameters 5,208,460 of 276,389,132
continuing from /kaggle/input/datasets/patelpreeet/lorma-p5-pt/p5_1_lt_lorma_best_trainable.pt
  starting point (loaded weights): dev macro-F1 0.3268


train:   0%|          | 0/2672 [00:00<?, ?it/s]

  2.30s/step, projected training time 1h51m
  step 222/2672 (epoch 0.08) dev macro-F1 0.3161 best 0.3268 | loss 0.2829 grad 0.55 lr x1 | prob std 0.1665 skipped 0 | ETA 1h32m
  step 444/2672 (epoch 0.17) dev macro-F1 0.3235 best 0.3268 | loss 0.2882 grad 0.56 lr x1 | prob std 0.1625 skipped 0 | ETA 1h24m
  step 666/2672 (epoch 0.25) dev macro-F1 0.3268 best 0.3268 | loss 0.2791 grad 0.55 lr x1 | prob std 0.1643 skipped 0 | ETA 1h15m
  step 888/2672 (epoch 0.33) dev macro-F1 0.3261 best 0.3268 | loss 0.2823 grad 0.55 lr x1 | prob std 0.1638 skipped 0 | ETA 1h07m
  step 1110/2672 (epoch 0.42) dev macro-F1 0.3287 best 0.3287 | loss 0.2848 grad 0.55 lr x1 | prob std 0.1648 skipped 0 | ETA 0h59m
  step 1332/2672 (epoch 0.50) dev macro-F1 0.3353 best 0.3353 | loss 0.2856 grad 0.55 lr x1 | prob std 0.1675 skipped 0 | ETA 0h50m
  step 1554/2672 (epoch 0.58) dev macro-F1 0.3352 best 0.3353 | loss 0.2768 grad 0.54 lr x1 | prob std 0.1662 skipped 0 | ETA 0h42m
  step 1776/2672 (epoch 0.66) dev ma